# 🆘 GemmAid — Acil Triaj Sistemi
## Kaggle Gemma 4 Good Hackathon 2026

**GemmAid**, kriz ve acil anlarda insanların ihtiyacını kendi dillerinde serbetçe ifade etmesini sağlar; Gemma 4 bu ifadeyi yapılandırılmış triaj verisine çevirerek doğru yardımın doğru kişiye koordineli şekilde ulaşmasını sağlar.

---

### Bu Notebook'ta:
1. **Bağımlılık kurulumu** ve Gemma 4 E4B yükleme (HuggingFace Transformers)
2. **4 dil triaj demosu** (TR/AR/FR/EN)
3. **Sonuç analizi** ve görselleştirme

### Neden Gemma 4?
- 140+ dil desteği (Arapça, Kürtçe, Fransızca triaj)
- Yapılandırılmış JSON çıktı (function calling)
- E4B edge mimarisi: CPU/GPU üzerinde çalışır
- Long context (256K): Zincirleme mesaj bağlamı

### Teknik Yığın:
- **Gemma 4 E4B** (google/gemma-4-e4b-it) — Google'ın açık kaynak modeli
- **HuggingFace Transformers** — resmi pipeline API
- **Kaggle GPU** veya CPU üzerinde çalışır

In [ ]:
# ── Hücre 1: Bağımlılık Kurulumu ────────────────────────────
import subprocess, sys

print('📦 Gerekli paketler kontrol ediliyor...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'transformers', 'accelerate', 'json-repair', '-q'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('✅ Paketler hazır')
else:
    print('⚠️  Uyarı:', result.stderr[:300])

In [ ]:
# ── Hücre 2: Import ve Konfigürasyon ────────────────────────
import json
import re
import time
import torch
from datetime import datetime, timezone

# Model konfigürasyonu
MODEL_ID   = 'google/gemma-4-e4b-it'   # Resmi Google Gemma 4 E4B
MAX_TOKENS = 512

# Cihaz tespiti
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Triaj sistem promptu
SYSTEM_PROMPT = """\
Sen GemmAid acil triaj asistanısın.
Kriz mesajlarını analiz ederek SADECE geçerli bir JSON nesnesi döndürürsün.
Başka hiçbir açıklama, markdown veya metin ekleme.

JSON formatı (tüm alanlar zorunlu):
{
  \"olay_tipi\": \"enkaz_alti|tibbi_acil|tahliye|kaynak_ihtiyaci|belirsiz\",
  \"aciliyet_skoru\": 1,
  \"konum_metni\": \"konum veya Belirtilmedi\",
  \"etkilenen_kisi_sayisi\": 1,
  \"semptomlar\": [\"semptom1\"],
  \"gerekli_ekip\": [\"tibbi|arama_kurtarma|tahliye|lojistik\"],
  \"kaynak_dil\": \"tr|ar|en|fr\",
  \"koordinator_notu\": \"Türkçe kısa açıklama\",
  \"vatandasa_yanit\": \"Kendi dilinde empati kuran güven verici mesaj\"
}

aciliyet_skoru: 1=kritik (hayat tehlikesi), 5=düşük öncelik
"""

ACIL_EMOJI = {1: '🔴', 2: '🟠', 3: '🟡', 4: '🟢', 5: '⚪'}

print('✅ Konfigürasyon yüklendi')
print(f'   Model  : {MODEL_ID}')
print(f'   Device : {DEVICE}')
print(f'   Max Tok: {MAX_TOKENS}')

In [ ]:
# ── Hücre 3: Yardımcı Fonksiyonlar ──────────────────────────

def parse_triage_json(text: str) -> dict:
    """LLM çıktısından JSON çıkar ve doğrula."""
    text = re.sub(r'```json\s*', '', text).strip()
    text = re.sub(r'```\s*', '', text)
    s = text.find('{')
    e = text.rfind('}') + 1
    if s != -1 and e > s:
        try:
            return json.loads(text[s:e])
        except json.JSONDecodeError:
            pass
    return {
        'olay_tipi': 'belirsiz',
        'aciliyet_skoru': 3,
        'koordinator_notu': f'Parse hatası: {text[:80]}'
    }


def display_triage_card(lang: str, message: str, result: dict, elapsed: float):
    """Triaj sonucunu formatlanmış olarak göster."""
    acil  = result.get('aciliyet_skoru', 3)
    emoji = ACIL_EMOJI.get(acil, '❓')
    
    print(f'\n{"─"*60}')
    print(f'🌐 Dil: {lang}  |  Mesaj: {message[:60]}...')
    print(f'─'*60)
    print(f'{emoji} Aciliyet Skoru  : {acil}/5')
    print(f'   Olay Tipi      : {result.get("olay_tipi", "?")}')
    print(f'   Konum          : {result.get("konum_metni", "Belirtilmedi")}')
    print(f'   Etkilenen Kişi : {result.get("etkilenen_kisi_sayisi", "?")}')
    print(f'   Gerekli Ekip   : {", ".join(result.get("gerekli_ekip", []))}')
    print(f'   Semptomlar     : {", ".join(result.get("semptomlar", []))}')
    print(f'   Tespit Dili    : {result.get("kaynak_dil", "?").upper()}')
    print(f'   Koordinatör    : {result.get("koordinator_notu", "-")}')
    print(f'   İşlem Süresi   : {elapsed:.1f}s')

print('✅ Yardımcı fonksiyonlar hazır')

In [ ]:
# ── Hücre 4: Model Yükleme (HuggingFace Transformers) ───────
from transformers import pipeline as hf_pipeline

print('🔄 Gemma 4 E4B yükleniyor...')
print('   (İlk yükleme 1-3 dakika sürebilir, model ~8GB indirilecek)')
print()

load_start = time.time()
pipe = hf_pipeline(
    task='any-to-any',
    model=MODEL_ID,
    device_map='auto',
    dtype='auto',
)
load_time = time.time() - load_start

print(f'✅ Model hazır! ({load_time:.1f} saniyede yüklendi)')
print(f'   Model   : {MODEL_ID}')
print(f'   Backend : HuggingFace Transformers')
print(f'   Device  : {DEVICE}')

In [ ]:
# ── Hücre 5: Demo Senaryoları ────────────────────────────────

SCENARIOS = [
    {
        'lang': 'Türkçe 🇹🇷',
        'context': 'Malatya Depremi — Enkaz Altı',
        'message': 'Komşumuz enkaz altında kaldı, Atatürk Caddesi 3. kat, nefes güçlüğü var, biz 3 kişiyiz'
    },
    {
        'lang': 'Arapça 🇸🇦',
        'context': 'Kahramanmaraş Depremi — Enkaz Altı',
        'message': 'جارنا عالق تحت الأنقاض، يتنفس بصعوبة، شارع أتاتورك، الطابق الثالث، نحن ثلاثة أشخاص'
    },
    {
        'lang': 'Fransızca 🇫🇷',
        'context': 'Sağlık Ocağı — Göçmen Hasta',
        'message': "J'ai une douleur intense dans la poitrine depuis 2 heures, j'ai du mal à respirer"
    },
    {
        'lang': 'İngilizce 🇬🇧',
        'context': 'Sel Felaketi — Tahliye',
        'message': 'House flooded, 2 children trapped on roof, water rising fast, Kemaliye district'
    },
]

print('=' * 60)
print('GemmAid — 4 Dil Triaj Demo')
print('Backend: HuggingFace Transformers + Gemma 4 E4B')
print('=' * 60)

all_results = []
total_start = time.time()

for i, scenario in enumerate(SCENARIOS, 1):
    lang    = scenario['lang']
    ctx     = scenario['context']
    message = scenario['message']
    
    print(f'\n[{i}/{len(SCENARIOS)}] {lang} — {ctx}')
    print(f'Analiz ediliyor...', end='', flush=True)
    
    # Resmi Gemma 4 chat format (system + user roller)
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user',   'content': [{'type': 'text', 'text': f'Kriz mesajı: {message}'}]},
    ]
    
    start = time.time()
    try:
        output = pipe(messages, max_new_tokens=MAX_TOKENS, return_full_text=False)
        raw = output[0]['generated_text'] if output else ''
        elapsed = time.time() - start
        print(f' ✅ ({elapsed:.1f}s)')
        
        result = parse_triage_json(raw)
        display_triage_card(lang, message, result, elapsed)
        
        all_results.append({
            'lang': lang,
            'context': ctx,
            'message': message,
            'result': result,
            'elapsed': elapsed,
            'success': True
        })
        
    except Exception as e:
        elapsed = time.time() - start
        print(f' ❌ ({elapsed:.1f}s)')
        print(f'   Hata: {e}')
        all_results.append({'lang': lang, 'error': str(e), 'success': False})

total_time = time.time() - total_start
successful = sum(1 for r in all_results if r.get('success'))

print()
print('=' * 60)
print(f'📊 Demo Tamamlandı!')
print(f'   Başarılı  : {successful}/{len(SCENARIOS)} senaryo')
print(f'   Toplam    : {total_time:.1f} saniye')
print(f'   Ortalama  : {total_time/len(SCENARIOS):.1f}s/triaj')
print(f'   Backend   : HuggingFace Transformers + Gemma 4 E4B')
print('=' * 60)

In [ ]:
# ── Hücre 6: Sonuç Görselleştirme ───────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('GemmAid — Triaj Sonuçları (Gemma 4 E4B)', 
             fontsize=14, fontweight='bold')

successful_results = [r for r in all_results if r.get('success')]

# 1. Aciliyet Skorları
ax1 = axes[0]
langs = [r['lang'].split()[0] for r in successful_results]
scores = [r['result'].get('aciliyet_skoru', 3) for r in successful_results]
colors = {1: '#FF4444', 2: '#FF8C00', 3: '#FFD700', 4: '#4CAF50', 5: '#9E9E9E'}
bar_colors = [colors.get(s, '#9E9E9E') for s in scores]
bars = ax1.bar(langs, scores, color=bar_colors, edgecolor='white', linewidth=2)
ax1.set_title('Aciliyet Skoru (1=Kritik, 5=Düşük)', fontsize=11)
ax1.set_ylim(0, 5.5)
ax1.set_ylabel('Aciliyet Skoru')
for bar, score in zip(bars, scores):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             str(score), ha='center', fontweight='bold', fontsize=12)

# 2. İşlem Süreleri
ax2 = axes[1]
elapsed = [r['elapsed'] for r in successful_results]
ax2.barh(langs, elapsed, color='#2196F3', edgecolor='white', linewidth=2)
ax2.set_title('İşlem Süresi (saniye)', fontsize=11)
ax2.set_xlabel('Süre (saniye)')
for i, (e, lang) in enumerate(zip(elapsed, langs)):
    ax2.text(e + 0.1, i, f'{e:.1f}s', va='center', fontweight='bold')

# 3. Olay Tipi Dağılımı
ax3 = axes[2]
olay_types = [r['result'].get('olay_tipi', 'belirsiz') for r in successful_results]
olay_labels = {
    'enkaz_alti': 'Enkaz Altı',
    'tibbi_acil': 'Tıbbi Acil',
    'tahliye': 'Tahliye',
    'kaynak_ihtiyaci': 'Kaynak İhtiyacı',
    'belirsiz': 'Belirsiz'
}
unique_types = list(set(olay_types))
counts = [olay_types.count(t) for t in unique_types]
pie_labels = [olay_labels.get(t, t) for t in unique_types]
pie_colors = ['#FF4444', '#2196F3', '#4CAF50', '#FF8C00', '#9E9E9E']
wedges, texts, autotexts = ax3.pie(
    counts, labels=pie_labels, colors=pie_colors[:len(unique_types)],
    autopct='%1.0f%%', startangle=90
)
ax3.set_title('Olay Tipi Dağılımı', fontsize=11)

plt.tight_layout()
plt.savefig('gemmaid_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Grafik kaydedildi: gemmaid_results.png')

In [ ]:
# ── Hücre 7: Özet JSON Çıktı ────────────────────────────────
print('📄 Tüm Triaj Sonuçları (JSON):')
print('=' * 60)

summary = {
    'system': 'GemmAid',
    'backend': 'HuggingFace Transformers + Gemma 4 E4B',
    'timestamp': datetime.now(timezone.utc).isoformat() + 'Z',
    'scenarios_total': len(SCENARIOS),
    'scenarios_success': sum(1 for r in all_results if r.get('success')),
    'total_time_seconds': round(total_time, 2),
    'results': [
        {
            'lang': r['lang'],
            'context': r.get('context', ''),
            'triage': r.get('result', {}),
            'elapsed_seconds': round(r.get('elapsed', 0), 2)
        }
        for r in all_results if r.get('success')
    ]
}

print(json.dumps(summary, ensure_ascii=False, indent=2))

## ✅ Demo Tamamlandı!

### Ne Gösterdik?
- Gemma 4 E4B modelinin **GPU/CPU üzerinde** HuggingFace Transformers ile çalışabildiğini
- **4 farklı dilde** (TR/AR/FR/EN) kriz mesajlarını analiz edebildiğini
- **Yapılandırılmış triaj verisi** (olay tipi, aciliyet, konum, ekip, vatandaş yanıtı) ürettiğini
- **Resmi Gemma 4 chat template** kullanılarak güvenilir JSON çıktısı alındığını

### Sonraki Adımlar
- [GitHub Repo](https://github.com/your-username/kaggle-gemma4_gemmaid)
- Telegram Bot ile gerçek zamanlı kullanım
- Koordinatör Dashboard (Gradio)
- Kapsamlı multi-agent koordinasyon sistemi

---
*GemmAid | Kaggle Gemma 4 Good Hackathon 2026 | Son Teslim: 18 Mayıs 2026*